<a href="https://colab.research.google.com/github/OmarArias-Gaguancela/QuickProt/blob/main/QuickProt_ID_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/OmarArias-Gaguancela/LOGOS_OAG/main/QuickProt ID Search.jpg" width="180" height="100">

# **QuickProt-ID Search**: This pipeline aims to search Protein IDs in the UniProt database based on a list of gene names

## **Requirements:**
## 1. Create a folder in MyDrive where analyzed data will be stored
## 2. List the target genes in a column of a spreadsheet in CSV format. The heading of the column should be titled **"Gene name"**
## 3. Save the CSV file in the recently created folder

## **Output:**
### 1. The spreadsheet named **"Target_gene_ProteinID_list.csv"** will be saved in the folder created by the user

In [5]:
import nbformat

notebook_path = 'QuickProt-ID Search.ipynb'  # change to your filename

with open(notebook_path, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

if 'widgets' in nb.metadata:
    for key, val in nb.metadata['widgets'].items():
        if 'state' not in val:
            val['state'] = {}

for cell in nb.cells:
    if 'widgets' in cell.metadata:
        for key, val in cell.metadata['widgets'].items():
            if 'state' not in val:
                val['state'] = {}

with open(notebook_path, 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)


In [ ]:
#@title ##**Mounts drive and changes the directory to the folder previously created**

from google.colab import drive
import os
from IPython.display import display
import ipywidgets as widgets

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")

instruction_label = widgets.HTML(
    value="<b>Tip:</b> To search for folders within folders, use <code>/</code> to separate them. For example: <code>Folder1/Subfolder1</code>"
)

folder_input = widgets.Text(
    value='',
    placeholder='Enter part of the folder name (e.g., Folder/Subfolder)',
    description='Search:',
    layout=widgets.Layout(width='50%')
)

button_style = {
    'font_weight': 'bold',
    'button_color': '#4CAF50',
    'font_size': '16px'
}
button_layout = widgets.Layout(width='auto', height='40px')

search_button = widgets.Button(
    description='Search',
    style=button_style,
    layout=button_layout
)

search_message_output = widgets.Output()

folder_dropdown = widgets.Dropdown(
    options=[],
    description='Select:',
    layout=widgets.Layout(width='50%')
)

confirm_button = widgets.Button(
    description='Set Directory',
    style=button_style,
    layout=button_layout
)

output = widgets.Output()

def search_folders(b):
    base_path = '/content/drive/MyDrive'
    matches = []
    search_term = folder_input.value.lower()

    with search_message_output:
        search_message_output.clear_output()
        print("Searching for folders... Please be patient, this may take up to 30 seconds.")

    if search_term:
        for root, dirs, files in os.walk(base_path):
            for d in dirs:
                full_path = os.path.join(root, d)
                relative_path = os.path.relpath(full_path, base_path)
                if search_term in relative_path.lower():
                    matches.append(full_path)
        folder_dropdown.options = matches if matches else ['No matches found']

def set_directory(b):
    selected_path = folder_dropdown.value
    if selected_path and selected_path != 'No matches found':
        os.chdir(selected_path)
        with output:
            output.clear_output()
            print(f"Current directory: {os.getcwd()}")

search_button.on_click(search_folders)
confirm_button.on_click(set_directory)

display(
    instruction_label,
    folder_input,
    search_button,
    search_message_output,
    folder_dropdown,
    confirm_button,
    output
)

Mounted at /content/drive


HTML(value='<b>Tip:</b> To search for folders within folders, use <code>/</code> to separate them. For example…

Text(value='', description='Search:', layout=Layout(width='50%'), placeholder='Enter part of the folder name (…

Button(description='Search', layout=Layout(height='40px', width='auto'), style=ButtonStyle(button_color='#4CAF…

Output()

Dropdown(description='Select:', layout=Layout(width='50%'), options=(), value=None)

Button(description='Set Directory', layout=Layout(height='40px', width='auto'), style=ButtonStyle(button_color…

Output()

In [ ]:
#@title **Install necessary packages**
import os

os.system('pip install unipressed > /dev/null 2>&1')
print("✅ Packages were installed successfully.")


✅ Packages were installed successfully.


In [ ]:
#@title **Run UniProt Search**
from unipressed import IdMappingClient
import time
import pandas as pd
import os

output_dir = 'TABLES'
os.makedirs(output_dir, exist_ok=True)

file_path = 'Target_gene_list.csv'
gene_data = pd.read_csv(file_path)

gene_names = gene_data['Gene name'].tolist()

request = IdMappingClient.submit(
    source="GeneCards", dest="UniProtKB", ids=set(gene_names)
)

time.sleep(1.0)

results_df = pd.DataFrame(list(request.each_result()))
column_mapping = {
    'from': 'Gene name', 'to': 'Protein IDs'
}
results_df.rename(columns=column_mapping, inplace=True)

results_df.sort_values(by='Gene name', inplace=True)

results_df.to_csv('TABLES/Target_gene_ProteinID_list.csv', index=False)
print("File was saved as: TABLES/Target_gene_ProteinID_list.csv")
results_df.head()

File was saved as: TABLES/Target_gene_ProteinID_list.csv


,Gene name,Protein IDs
3,ACTB,P60709
26,ACTL6A,O96019
5,ACTL6B,O94805
7,ARID1A,O14497
10,ARID1B,Q8NFD5
